In [3]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langchain.agents import create_agent


# .env 파일에서 환경 변수 로드
load_dotenv()

# 모델 선언
model = init_chat_model("gpt-4o-mini")

In [4]:
from dataclasses import dataclass
from typing import TypedDict
from langgraph.store.memory import InMemoryStore

# 1. 실행 컨텍스트 정의 (누가 실행하는지 식별)
@dataclass
class Context:
    user_id: str
    app_name: str

# 2. LLM이 추출해야 할 정보의 구조를 정의
class UserInfo(TypedDict):
    personal_information : str
    preference: str
# LLM이 추출해야 할 정보의 구조를 정의 (사용자의 request로 부터 추출해야 할 정보의 구조를 정의)
# 즉, LLM이 request로 부터 정보를 추출한다음, 해당 정보를 Tool에 전달해야하는 경우 전달한다

# 3. 빈 Store 초기화
store = InMemoryStore()


`TypedDict`와 `Pydantic`의 가장 큰 차이는 **"데이터 검증(Validation)을 하느냐 마느냐"**에 있습니다.

비유하자면, `TypedDict`는 **"이름표"**이고, `Pydantic`은 **"검사원"**입니다.

---

### 1. 주요 차이점 비교

| 특징 | `TypedDict` (파이썬 내장) | `Pydantic` (외부 라이브러리) |
| :--- | :--- | :--- |
| **실행 시 검증** | **안 함.** (타입이 틀려도 에러 안 남) | **함.** (타입이 틀리면 즉시 에러 발생) |
| **타입 변환** | 안 함. (문자열을 넣으면 문자열 그대로임) | **함.** (숫자 자리에 "10"을 넣으면 `10`으로 변환) |
| **정체** | 단순한 **딕셔너리(`dict`)** | 새로운 **객체(Object)** |
| **속도** | 매우 빠름 (검증 과정이 없으므로) | 상대적으로 느림 (검증 과정이 있으므로) |

---

### 2. 예시로 보는 차이

#### ① `TypedDict` (검사 안 함)
```python
from typing import TypedDict

class User(TypedDict):
    age: int

# 에러가 안 납니다! (타입 힌트일 뿐이라서)
user = User(age="스무살") 
print(user["age"]) # "스무살" (문자열 그대로 출력됨)
```

#### ② `Pydantic` (엄격하게 검사함)
```python
from pydantic import BaseModel

class User(BaseModel):
    age: int

# 에러가 발생하거나 자동 변환됩니다!
user1 = User(age="20") # "20"을 정수 20으로 자동 변환 (성공)
user2 = User(age="스무살") # 정수로 바꿀 수 없으므로 ValidationError 발생! (실패)
```

---

### 3. 언제 무엇을 쓰나요?

*   **`TypedDict`를 쓰는 경우:**
    *   성능이 매우 중요할 때.
    *   단순히 딕셔너리에 어떤 키가 들어있는지 **코드 작성 중에만** 참고하고 싶을 때.
    *   LLM의 응답을 가볍게 받아서 다른 곳으로 넘길 때.

*   **`Pydantic`을 쓰는 경우:**
    *   데이터의 **정확성**이 매우 중요할 때 (예: 나이가 음수면 안 됨).
    *   LLM이 내뱉은 불안정한 데이터를 **안전하게 정제**하고 싶을 때.
    *   복잡한 중첩 구조나 기본값 설정이 필요할 때.

### 요약
- **`TypedDict`**: "이 딕셔너리에는 `age`라는 키가 있을 거야"라고 **알려주기만** 함. (실제로 틀려도 상관없음)
- **`Pydantic`**: "이 데이터는 반드시 `int`여야 해!"라고 **강제하고 검사**함. (틀리면 바로 에러!)

최근 LangChain에서는 LLM의 답변을 구조화할 때 **Pydantic**을 더 권장하는 추세입니다. (LLM이 가끔 이상한 값을 주므로 검증이 꼭 필요하기 때문입니다.)

In [ ]:
import uuid
from langchain_core.runnables import RunnableConfig
from langchain.tools import tool, ToolRuntime

# --- 조회 도구 ---
@tool
def get_user_info(runtime) -> str:
    """
    현재 사용자의 정보 조회 (시스템 내부용 도구)
    """
    user_id = runtime.context.user_id
    app = runtime.context.app_name

    # 해당 네임스페이스의 모든 메모리 검색
    memories = runtime.store.search((user_id, app))

    if not memories:
        return "기록된 정보 없음"

    results = []
    for item in memories:
        # 저장된 데이터 구조(UserInfo)에 맞춰 필드 확인
        data = item.value
        print('data', data)
        if "personal_information" in data: # 딕셔너리 key 중에 personal_information이 있으면 
            results.append(f"- 개인정보: {data['personal_information']}")
        if "preference" in data:
            results.append(f"- 선호도: {data['preference']}")

    return "\n".join(results) if results else "데이터 형식 불일치로 읽을 수 없음"

In [17]:
from langchain.tools import ToolRuntime

# --- 저장 도구 ---
@tool
def save_user_info(user_info: UserInfo, runtime: ToolRuntime) -> str:
    """
    사용자의 정보를 저장하거나 업데이트
    """
    # 1. 실행 컨텍스트에서 user_id 가져오기
    user_id = runtime.context.user_id
    app = runtime.context.app_name
    store = runtime.store

    # 2. Store에 데이터 저장 (UUID를 생성하여 계속 누적)
    memory_key = str(uuid.uuid4())
    store.put((user_id, app), memory_key, user_info)

    return f"정보가 안전하게 저장되었습니다. (ID: {memory_key})"

--------

도구(`@tool`)로 정의된 함수가 전달받는 파라미터가 서로 다른 이유는 **"AI가 채워줘야 할 정보"**와 **"시스템이 자동으로 채워주는 정보"**가 섞여 있기 때문입니다.

LangChain의 에이전트 환경에서 도구가 받을 수 있는 파라미터는 크게 두 종류로 나뉩니다.

---

### 1. AI(LLM)가 채워주는 파라미터
사용자의 질문에서 정보를 추출하여 AI가 직접 값을 넣어주는 파라미터입니다.

*   **예시 (`save_user_info`):**
    ```python
    def save_user_info(user_info: UserInfo, runtime):
    ```
    여기서 `user_info`는 AI가 사용자의 말을 듣고 "이름은 Alice, 차를 좋아함"이라는 데이터를 구조화해서 넣어주는 값입니다.

### 2. 시스템이 자동으로 채워주는 파라미터 (Injected Parameters)
AI는 이 파라미터의 존재를 모르거나 직접 값을 넣지 않습니다. 대신 에이전트 실행 환경(Runtime)이 자동으로 주입해 줍니다.

*   **`runtime`**: 현재 실행 중인 에이전트의 **컨텍스트(Context)**나 **저장소(Store)**에 접근하기 위해 사용합니다.
*   **왜 `get_user_info`에는 이것만 있나요?**
    ```python
    def get_user_info(runtime):
    ```
    사용자 정보를 조회할 때는 사용자가 별도의 정보를 줄 필요가 없습니다. 그냥 "내 정보 보여줘"라고만 하면 되죠. 하지만 함수 내부에서는 **"지금 질문한 사람이 누구인지(user_id)"**를 알아야 DB에서 데이터를 꺼낼 수 있습니다. 이 `user_id`를 가져오기 위해 `runtime` 객체가 필요한 것입니다.

---

### 3. 도구가 전달받을 수 있는 파라미터 규칙

`@tool` 데코레이터를 사용할 때 다음과 같은 것들을 파라미터로 쓸 수 있습니다.

1.  **일반 변수 (AI가 채움):** `query: str`, `count: int`, `user_info: UserInfo` 등. (타입 힌트를 적어주면 AI가 더 잘 이해합니다.)
2.  **특수 변수 (시스템이 채움):** 
    *   **`runtime`**: 컨텍스트, 스토어 등에 접근할 때 사용 (가장 많이 쓰임).
    *   **`config`**: `RunnableConfig` 객체로, 콜백이나 태그 정보 등에 접근할 때 사용.
    *   **`tool_call_id`**: 현재 실행 중인 도구 호출의 고유 ID.

---

### 4. 왜 두 함수의 파라미터가 다른가요? (결론)

*   **`save_user_info`**: **"무엇을 저장할지"** 알아야 하므로 `user_info`가 필요하고, **"누구의 저장소에 넣을지"** 알아야 하므로 `runtime`이 필요합니다.
*   **`get_user_info`**: 사용자가 줄 정보는 따로 없습니다. 하지만 **"누구의 정보를 가져올지"**는 알아야 하므로 `runtime`만 있으면 충분합니다.

### 요약
- **AI가 주는 것:** 함수 실행에 필요한 **데이터** (`user_info`)
- **시스템이 주는 것:** 함수 실행에 필요한 **환경 정보** (`runtime`)
- 도구는 이 두 가지를 조합해서 자신이 할 일을 수행하게 됩니다.

----------

## 🛠️ LangChain `@tool` 인자 결정 및 전달 원리

LangChain의 `@tool` 데코레이터를 사용할 때, 함수의 인자가 어떻게 결정되고 전달되는지에 대해 설명해 드리겠습니다.

**결론부터 말씀드리면:** LLM(모델)이 함수의 **"이름"**, **"설명(Docstring)"**, 그리고 **"인자의 타입 힌트"**를 보고 어떤 값을 넣을지 스스로 판단하여 전달합니다.

---

### 1. 명시적 인자가 있는 경우
> 예시: `02-08.agent-basic.py`

```python
@tool
def get_weather(city: str) -> str:
    """특정 도시의 현재 날씨를 가져옵니다."""
    return f"{city}의 날씨는 맑고 22도입니다."
```

*   **LLM의 판단:** 모델은 이 도구의 설명을 보고 "아, `city`라는 인자에 도시 이름을 넣어야겠구나"라고 이해합니다.
*   **전달 방식:** 사용자가 "서울 날씨 어때?"라고 물으면, LLM은 내부적으로 `{"city": "Seoul"}`이라는 JSON 데이터를 생성하고, LangChain은 이를 `get_weather(city="Seoul")` 형태로 호출합니다.
*   **핵심:** `city: str`이라는 **타입 힌트**와 `특정 도시의 현재 날씨를 가져옵니다`라는 **설명**이 LLM에게 가이드 역할을 합니다.

---

### 2. 특수한 인자가 있는 경우
> 예시: `get_user_info(runtime)` 함수

이 `runtime` 인자에 무엇이 전달될지는 프레임워크의 **'인자 주입(Injection)'** 규칙에 의해 결정됩니다.

1.  **LLM은 이 인자의 존재를 모름:** `@tool` 데코레이터가 LLM에게 보내는 도구 설명서에서 `runtime` 같은 특수 인자는 자동으로 제외합니다.
2.  **시스템이 자동 주입:** 프레임워크(LangGraph/LangChain)가 도구를 실행할 때, 인자 이름이 `runtime`인 것을 보고 자신이 관리하는 시스템 객체(State, Store 등)를 **자동으로 끼워 넣어** 호출합니다.

---

> **요약:** 
> - 일반 인자(`city`, `user_info` 등)는 **AI**가 채워줍니다.
> - 특수 인자(`runtime`, `config` 등)는 **시스템**이 자동으로 채워줍니다.

----------

In [18]:
from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-nano",
    tools=[get_user_info, save_user_info],
    store=store,         # 에이전트에 store 연결
    context_schema=Context
)


In [19]:
response1 = agent.invoke(
    {"messages": [{"role": "user", "content": "내 이름은 이제 'Alice'야. 커피보단 차를 좋아해"}]},
    context=Context(user_id="user_001", app_name="personal_assistant")
)
print(response1)
print(response1["messages"][-1].content)


/Users/dohyunkim/Documents/langchain-for-ai-agent/.venv/lib/python3.13/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=Context(user_id='user_001...me='personal_assistant'), input_type=Context])
  return self.__pydantic_serializer__.to_python(


{'messages': [HumanMessage(content="내 이름은 이제 'Alice'야. 커피보단 차를 좋아해", additional_kwargs={}, response_metadata={}, id='c7fc1727-2ffa-449e-b422-0153f361b0dd'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 609, 'prompt_tokens': 181, 'total_tokens': 790, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 576, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DMB3PP7xO58jLFaFWfwgi9D2djAwV', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d1541-1121-7e92-a7f1-409f5d2d9843-0', tool_calls=[{'name': 'save_user_info', 'args': {'user_info': {'personal_information': 'Alice', 'preference': 'tea'}}, 'id': 'call_vlCcCQM4PaUlH0k8bHvnxL8M', 'type': 'tool_call'}], invalid_tool_calls=

In [16]:
response2 = agent.invoke(
    {"messages": [{"role": "user", "content": "나에 대해 아는 정보 말해줘"}]},
    context=Context(user_id="user_001", app_name="personal_assistant")
)
print(response2)
print(response2["messages"][-1].content)


data {'personal_information': 'Alice', 'preference': 'tea'}
data {'personal_information': 'Name: Alice', 'preference': 'tea over coffee'}
data {'personal_information': 'name: Alice', 'preference': 'prefers tea to coffee'}
{'messages': [HumanMessage(content='나에 대해 아는 정보 말해줘', additional_kwargs={}, response_metadata={}, id='d0a6b0ff-02b7-4a58-afa6-1307e5a579c2'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 408, 'prompt_tokens': 176, 'total_tokens': 584, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 384, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DMAyILZz7dZInKGdIVGkJv6UJYFED', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d153c-35ab-7f61-96b8-375e59abe

`03/10.ipynb`와 `03/11.ipynb`의 가장 핵심적인 차이는 **"누가 정보를 가져오느냐"**와 **"언제 가져오느냐"**에 있습니다.

**"토큰 낭비"** 문제를 중심으로 두 방식의 차이점을 명확히 비교해 보겠습니다.

---

### 1. 03/10.ipynb 방식: "미들웨어가 무조건 주입" (Push 방식)
이 방식은 LLM이 물어보기도 전에 미들웨어가 과거의 모든 기억을 꺼내서 프롬프트에 미리 넣어주는 방식입니다.

*   **동작:** 사용자가 "안녕"이라고만 해도, 미들웨어가 DB에서 "이 사람은 Alice고, 차를 좋아함"이라는 정보를 찾아 프롬프트 맨 앞에 붙여서 모델에게 보냅니다.
*   **문제점 (토큰 낭비):** 
    *   사용자가 "오늘 날씨 어때?"라고 물어볼 때처럼 **사용자 정보가 전혀 필요 없는 대화**에서도 매번 과거 기억을 프롬프트에 포함합니다.
    *   대화가 길어지고 기억해야 할 정보가 많아질수록, 매 요청마다 수천 토큰의 과거 데이터를 LLM에게 계속 보내야 하므로 비용이 급증합니다.

---

### 2. 03/11.ipynb 방식: "모델이 필요할 때만 호출" (Pull 방식)
이 방식은 LLM에게 `get_user_info`라는 **도구(Tool)**를 쥐여주고, 모델 스스로 판단하게 하는 방식입니다.

*   **동작:** 
    1. 사용자가 "안녕"이라고 하면, 모델은 사용자 정보가 필요 없다고 판단하고 그냥 대답합니다. (토큰 절약!)
    2. 사용자가 "내가 좋아하는 음료로 추천해줘"라고 하면, 모델은 **"아, 이 사람의 취향을 알아야겠네? `get_user_info` 도구를 써야지!"**라고 판단하여 그때만 정보를 가져옵니다.
*   **장점 (효율성):**
    *   **필요할 때만:** 정보가 필요한 시점에만 도구를 호출하므로 평소에는 토큰을 낭비하지 않습니다.
    *   **선택적 추출:** 수많은 정보 중에서도 현재 질문과 관련된 정보만 골라서 가져올 수 있도록 설계할 수 있습니다.

---

### 3. 핵심 차이점 요약

| 비교 항목 | 03/10 (미들웨어 주입) | 03/11 (도구 호출 방식) |
| :--- | :--- | :--- |
| **주도권** | **시스템(미들웨어)**이 강제로 주입 | **LLM(모델)**이 스스로 판단하여 요청 |
| **토큰 사용** | **매 요청마다** 과거 기억 전체 포함 (비효율) | **필요한 순간에만** 도구 결과 포함 (효율적) |
| **구현 방식** | `before_agent` 훅에서 `state` 수정 | `@tool` 정의 및 에이전트에 등록 |
| **비유** | 시험 공부할 때 **모든 교과서를 통째로** 들고 들어감 | 시험 보다가 모르는 게 나오면 **필요한 페이지만** 찾아봄 |

### 결론
`03/11.ipynb` 방식은 LLM에게 **"기억을 조회할 수 있는 능력(도구)"**을 부여한 것입니다. 이를 통해 에이전트는 훨씬 더 경제적이고 똑똑하게 과거의 정보를 활용할 수 있게 되었습니다. 

"자기 마음대로 도구를 호출한다"고 느껴지는 현상이 바로 이 **"모델의 자율적인 판단에 의한 정보 조회"** 과정입니다.